# LLM Quantization with GPTQ (Generative Pre-Trained Quantization)

This notebook quantizes `Qwen/Qwen2.5-0.5B-Instruct` to **4-bit GPTQ** format using the `auto-gptq` library.

**Pipeline:**
1. Install dependencies
2. Load tokenizer + model via AutoGPTQ
3. Load calibration texts from medquad.csv
4. Run GPTQ quantization
5. Save quantized model + tokenizer
6. Reload and verify with inference

**AWQ vs GPTQ key differences:**
- GPTQ uses second-order (Hessian) weight error minimization; AWQ uses activation-aware scaling
- GPTQ calibration data must be **pre-tokenized** into `input_ids` tensors (unlike AWQ which takes raw strings)
- GPTQ generally achieves slightly higher accuracy; AWQ is faster to quantize
- Both produce 4-bit int weights and are loaded via their respective libraries

**Requirements:** GPU with ≥8GB VRAM (T4 on Colab is sufficient)

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
# auto-gptq : GPTQ quantization library (maintained by PanQiWei / HuggingFace)
# transformers: model/tokenizer loading and inference
# accelerate : device management helpers used internally by auto-gptq
# optimum    : HuggingFace integration layer for GPTQ — needed for from_quantized()
!pip install -q -U auto-gptq transformers accelerate optimum

In [ ]:
# ── Cell 2: Imports ────────────────────────────────────────────────────────────
import os
import gc
import torch
import pandas as pd
from auto_gptq import AutoGPTQForCausalLM, BaseQuantizeConfig   # GPTQ model wrapper + config
from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM

# Suppress tokenizer parallelism warning in notebook environments
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ── Sanity-check the environment ──────────────────────────────────────────────
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version    : {torch.version.cuda}")
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. GPTQ quantization requires a CUDA GPU.")

In [ ]:
# ── Cell 3: Configuration ──────────────────────────────────────────────────────

# Source model to quantize
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

# Output directory for the quantized model
OUT_DIR = "qwen2.5-0.5b-gptq"

# GPTQ quantization config via BaseQuantizeConfig
# bits          : quantize weights to 4 bits
# group_size    : number of weights per quantization group (128 is standard;
#                 smaller = better quality, larger file; -1 = per-column)
# desc_act      : use activation ordering (True = better quality, slower quant)
#                 set False on T4 to avoid triton kernel issues
QUANT_CONFIG = BaseQuantizeConfig(
    bits=4,
    group_size=128,
    desc_act=False,       # set True for higher quality if your GPU supports it
)

In [ ]:
# ── Cell 4: Clear GPU memory before loading ────────────────────────────────────
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()
print("GPU memory cleared.")

In [ ]:
# ── Cell 5: Load tokenizer ─────────────────────────────────────────────────────
print(f"Loading tokenizer from '{BASE_MODEL}'...")
tok = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    use_fast=True,           # Use the Rust-based fast tokenizer
    trust_remote_code=True,
)

# GPTQ's calibration loop batches sequences and needs a pad token.
# Qwen2.5 has eos_token; fall back to it if pad is unset.
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

print("Tokenizer loaded.")

In [ ]:
# ── Cell 6: Load model via AutoGPTQ ───────────────────────────────────────────
# AutoGPTQForCausalLM.from_pretrained() loads the fp16 base model and
# attaches GPTQ-specific quantization hooks ready for .quantize().
#
# Key arguments:
#   quantize_config   : BaseQuantizeConfig with bits, group_size, desc_act
#   low_cpu_mem_usage : load weights shard-by-shard to avoid CPU OOM
#   torch_dtype       : load in fp16 to fit in GPU VRAM before quantization
#   device_map        : pin everything to GPU 0
print(f"Loading model '{BASE_MODEL}' via AutoGPTQ...")
mdl = AutoGPTQForCausalLM.from_pretrained(
    BASE_MODEL,
    quantize_config=QUANT_CONFIG,
    low_cpu_mem_usage=True,
    torch_dtype=torch.float16,
    trust_remote_code=True,
    device_map="auto" if torch.cuda.is_available() else "cpu",
)

# Set to eval mode: disables dropout so activation statistics during
# calibration are clean and representative.
mdl.eval()
print("Model loaded and set to eval mode.")

In [ ]:
# ── Cell 7: Calibration data ───────────────────────────────────────────────────
# GPTQ needs pre-tokenized input_ids — unlike AWQ which accepts raw strings.
# Each calibration sample must be a dict with at least an 'input_ids' key
# (and optionally 'attention_mask') as a 1D or 2D tensor.
#
# Steps:
#   1. Load CSV and extract the 'answer' text column
#   2. Tokenize each text with padding + truncation
#   3. Convert to list[dict] of {input_ids, attention_mask}
print("Preparing calibration data...")

df = pd.read_csv("/content/medquad.csv")
print("CSV columns:", df.columns.tolist())

calib_texts = (
    df["answer"]      # use the answer column as calibration text
    .dropna()         # drop NaN rows
    .astype(str)      # ensure plain strings
    .head(100)        # 100 samples — good balance of quality vs speed
    .tolist()
)

# Tokenize each text into a dict of tensors.
# GPTQ expects a list[dict] where each dict has 'input_ids' (and optionally
# 'attention_mask') as 1-D or [1, seq_len] tensors.
CALIB_DATA = [
    tok(
        text,
        return_tensors="pt",          # return PyTorch tensors
        padding="max_length",         # pad all samples to the same length
        truncation=True,              # truncate samples longer than max_length
        max_length=128,               # keep memory low; increase for richer calibration
    )
    for text in calib_texts
]

print(f"{len(CALIB_DATA)} calibration samples ready.")
print(f"Each sample keys : {list(CALIB_DATA[0].keys())}")
print(f"input_ids shape  : {CALIB_DATA[0]['input_ids'].shape}")

In [ ]:
# ── Cell 8: Run GPTQ quantization ─────────────────────────────────────────────
# .quantize() performs three steps internally:
#   1. Forward pass on calib_data to compute per-layer Hessian matrices
#      (second-order information about weight sensitivity)
#   2. Use the Hessian to find optimal rounding for each weight group
#      (minimizes reconstruction error layer by layer)
#   3. Pack weights into int4 format using the GPTQ kernel
#
# Parameters:
#   examples         : list[dict] of pre-tokenized {input_ids, attention_mask}
#   batch_size       : how many samples to process at once (1 = safest for T4)
#   use_triton       : use Triton kernels for faster quantization (False = safer)
#   autotune_warmup_after_quantized : skip warm-up for speed
print("Starting GPTQ quantization...")

mdl.quantize(
    CALIB_DATA,
    batch_size=1,               # process one sample at a time to avoid OOM
    use_triton=False,           # set True if Triton is installed for ~2x speed
)

print("✅ Quantization complete!")

In [ ]:
# ── Cell 9: Save quantized model ──────────────────────────────────────────────
# save_quantized() writes the int4 weight tensors in safetensors format
# along with the quantize_config.json that records all GPTQ parameters.
# We also save the tokenizer so the output directory is self-contained.
print(f"Saving quantized model to '{OUT_DIR}'...")

mdl.save_quantized(OUT_DIR, use_safetensors=True)   # saves int4 weights + quantize_config.json
tok.save_pretrained(OUT_DIR)                         # saves tokenizer files

# Save the original model config (architecture metadata)
cfg = AutoConfig.from_pretrained(BASE_MODEL, trust_remote_code=True)
cfg.save_pretrained(OUT_DIR)

print(f"✅ Model saved to '{OUT_DIR}'")
print("Contents:")
for f in sorted(os.listdir(OUT_DIR)):
    size_mb = os.path.getsize(os.path.join(OUT_DIR, f)) / 1e6
    print(f"  {f:<45} {size_mb:>8.2f} MB")

In [ ]:
# ── Cell 10: Free memory before reload ────────────────────────────────────────
# Delete the quantization-time model object and flush GPU memory
# so we start the inference test with a clean slate.
del mdl
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()
print("Memory freed.")

In [ ]:
# ── Cell 11: Reload quantized model and run inference ─────────────────────────
# from_quantized() reads the int4 weights and the quantize_config.json,
# then reconstructs a model ready for fast GPTQ inference.
#
# Key arguments:
#   use_safetensors  : load from .safetensors files (must match how we saved)
#   disable_exllama  : set True if ExLlama kernel is unavailable (safer on Colab)
#   device_map       : put the model on GPU 0
print(f"Reloading quantized model from '{OUT_DIR}'...")

mdl_q = AutoGPTQForCausalLM.from_quantized(
    OUT_DIR,
    use_safetensors=True,
    trust_remote_code=True,
    disable_exllama=True,    # disable ExLlama v1 kernel; use standard GPTQ ops
    device_map="auto" if torch.cuda.is_available() else "cpu",
)
tok_q = AutoTokenizer.from_pretrained(OUT_DIR)
if tok_q.pad_token is None:
    tok_q.pad_token = tok_q.eos_token

# ── Run a simple text generation test ─────────────────────────────────────────
prompt = "What are risks of High Blood pressure"
inputs = tok_q(prompt, return_tensors="pt").to(
    next(mdl_q.parameters()).device
)

with torch.no_grad():
    output_ids = mdl_q.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,          # greedy decoding for reproducibility
        temperature=1.0,
        repetition_penalty=1.1,
    )

# Decode only the newly generated tokens (skip the prompt)
generated = tok_q.decode(
    output_ids[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True,
)

print("\n" + "=" * 60)
print(f"Prompt   : {prompt}")
print(f"Generated: {generated}")
print("=" * 60)
print("\n✅ GPTQ quantization pipeline complete!")